# Instructor Validators — Teaching the LLM to Self-Correct

**Week 1 | Notebook 2 of 4**

**What you'll learn:**
- Why validation fails with raw prompting
- `@field_validator` — custom validation logic
- Instructor retry loop — how it works under the hood
- `max_retries=3` — debugging retry traces
- ValidationError message injection — what the LLM sees
- Complex cross-field validation (domain matches email)
- Tenacity integration — exponential backoff

**Runtime:** ~40 minutes

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("01_instructor/02_validators.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  01_instructor/02_validators.ipynb
Task:      Auto-retry with validators
Calls:     ~15

With GPT-4o:       $0.23 USD
With GPT-4o-mini:  $0.02 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


In [2]:
from pydantic import BaseModel, ValidationError, field_validator

from src.config import USE_OLLAMA, get_instructor_client

client = get_instructor_client()  # provider from LLM_PROVIDER in .env (default: openai)

# Raw prompting often produces invalid output
# Example: model returns 'paul.krishai.com' instead of 'paul@krishai.com'
print("❌ Raw prompting has no built-in validation or retry")
print("✅ Instructor catches validation errors and feeds them back to the LLM")

❌ Raw prompting has no built-in validation or retry
✅ Instructor catches validation errors and feeds them back to the LLM


## 2. `@field_validator` — Custom Validation Logic

In [3]:
class EmailAddress(BaseModel):
    address: str
    domain: str
    is_corporate: bool

    @field_validator("address")
    @classmethod
    def must_be_valid_email(cls, v):
        if "@" not in v:
            raise ValueError(f"'{v}' is not a valid email — must contain '@'")
        return v.lower()

    @field_validator("domain")
    @classmethod
    def must_match_address(cls, v, values):
        if "address" in values.data:
            expected = values.data["address"].split("@")[1]
            if v != expected:
                raise ValueError(f"Domain '{v}' doesn't match email domain '{expected}'")
        return v


# Test the validator
try:
    email = EmailAddress(address="paul.krishai.com", domain="krishai.com", is_corporate=True)
except ValidationError as e:
    print("Validation caught the error:")
    print(e)

Validation caught the error:
1 validation error for EmailAddress
address
  Value error, 'paul.krishai.com' is not a valid email — must contain '@' [type=value_error, input_value='paul.krishai.com', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
